In [1]:
import os
import json
import asyncio
import datetime
from typing import Dict, List, Optional, Tuple
from pathlib import Path
import httpx
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich.markdown import Markdown
from rich.prompt import Prompt
from rich.rule import Rule
from rich.spinner import Spinner
from rich.columns import Columns
from rich import box

In [2]:
console = Console()

In [3]:
#Setting up environment
from google.colab import userdata
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

In [4]:
RED = "\033[31;1m"
GREEN = "\033[32;1m"
YELLOW = "\033[33;1m"
BLUE = "\033[34;1m"
PURPLE = "\033[35;1m"
CYAN = "\033[36;1m"
WHITE = "\033[37;1m"
ORANGE = "\033[38;5;208m"
RESET = "\033[0;0m"

In [5]:
#Token Pricing
MODEL_PRICING: Dict[str, Tuple[float, float]] = {
    "llama-3.1-8b-instant": (0.05, 0.08),
    "llama-3.3-70b-versatile": (0.59, 0.79),
    "meta-llama/llama-4-scout-17b-16e-instruct": (0.11, 0.34),
    "openai/gpt-oss-120b": (0.15, 0.60),
    "openai/gpt-oss-20b": (0.075, 0.30),
    "groq/compound": (0.15, 0.60),
    "groq/compound-mini": (0.075, 0.30),
    #"qwen/qwen3-32b": (0.29, 0.59),
    "qwen/qwen3.6-27b": (0.60, 3.0)
}

In [6]:
def price_for(model:str, in_tok:int, out_tok:int)->float:
  p = MODEL_PRICING.get(model, (0.5, 0.5))
  return (in_tok * p[0] + out_tok * p[1])/1_000_000

In [7]:
class GroqChatbot:
  DEFAULT_SYSTEM = (
      "You are a helpful, concise, and honest assistant."
      "Maintain context across the conversation and give accurate responses."
  )

  AVAILABLE_MODELS = list(MODEL_PRICING.keys())

  def __init__(self, model:str="openai/gpt-oss-20b", system_prompt:Optional[str]=None, max_history_tokens:int=6000, max_retries:int=3) -> None:
    self.api_key = GROQ_API_KEY
    if not self.api_key:
      console.print(Panel(
      "[bold red]GROQ API env variable not set.[/bold red]\n"
      "Run: [cyan]export GROQ_API='gsk...'[/cyan]",
      title = "[red]Configuration Error[/red]",
      border_style = "red"
    ))
      raise SystemExit(True)

    self.base_url = "https://api.groq.com/openai/v1/chat/completions"
    self.model = model if model in self.AVAILABLE_MODELS else self.AVAILABLE_MODELS[0]
    self.system = system_prompt or self.DEFAULT_SYSTEM
    self.max_history_tokens = max_history_tokens
    self.max_retries = max_retries

    self.conversation_history:List[Dict] = []
    self.session_start = datetime.datetime.now()

    self.total_in_tokens = 0
    self.total_out_tokens = 0
    self.total_cost_usd = 0.0

  #Helper Functions
  def _ts(self)->str:
    return datetime.datetime.now().strftime("%H:%M:%S")

  def _rough_token_count(self, text:str)->int:
    return max(1, len(text)//4)

  def _trim_history(self)->List[Dict]:
    budget, trimmed = self.max_history_tokens, []
    for msg in reversed(self.conversation_history):
      cost = self._rough_token_count(msg["content"])
      if cost > budget:
        break
      trimmed.insert(0, {"role":msg["role"], "content":msg["content"]})
      budget -= cost
    return trimmed

  def _add_to_history(self, role:str, content:str)->None:
    self.conversation_history.append({"role":role, "content":content, "ts":self._ts()})

  def _format_cost(self)->str:
    return f"$ {self.total_cost_usd:.6f}"

  #Model and System Management
  def set_model(self, model:str)->bool:
    if model in self.AVAILABLE_MODELS:
      self.model = model
      console.print(f"[bold green]Model switched to:[/][cyan]{model}[/]")
      return True
    console.print(f"[bold red] Unknown model:[/] {model}")
    return False

  def set_system_prompt(self, prompt:str)->None:
    self.system = prompt
    console.print(f"[bold green]System prompt updated[/]")

  #API Calling with retry and streaming
  async def stream_response(self, messages:List[Dict])->str:
    payload = {
        "model": self.model,
        "messages": messages,
        "max_tokens": 1024,
        "temperature": 0.7,
        "top_p": 0.95,
        "stream": True
    }

    headers = {
        "Authorization": f"Bearer {self.api_key}",
        "Content-Type": "application/json"
    }

    for attempt in range(1, self.max_retries + 1):
      try:
        async with httpx.AsyncClient() as client:
          async with client.stream("POST", self.base_url, json=payload, headers=headers) as response:
            response.raise_for_status()

            response_text = ""
            in_tokens_used = 0
            out_tokens_used = 0

            #Streaming Chunks
            print(f"{CYAN}{self.model}:", end="")

            async for line in response.aiter_lines():
              if not line.startswith("data:"):
                continue
              data = line[6:].strip()
              if data == "[DONE]":
                break
              try:
                chunk = json.loads(data)
              except json.JSONDecodeError:
                continue
              choices = chunk.get("choices", [])
              if choices:
                delta = choices[0].get("delta", {})
                content = delta.get("content", "")
                if content:
                  print(f"{CYAN}{content}", end="", flush=True)
                  response_text += content

                usage = chunk.get("x_groq", {}).get("usage", {})
              if usage:
                in_tokens_used = usage.get("prompt_tokens", 0)
                out_tokens_used = usage.get("completion_tokens", 0)

          print()
          #Fallback token estimation
          if in_tokens_used == 0:
            in_tokens_used = sum(self._rough_token_count(m["content"]) for m in messages)
            out_tokens_used += self._rough_token_count(response_text)

          self.total_in_tokens += in_tokens_used
          self.total_out_tokens += out_tokens_used
          cost = price_for(self.model, in_tokens_used, out_tokens_used)
          self.total_cost_usd += cost

          console.print(
              f"[dim] [{self._ts()}]"
              f"tokens: in-tokens: {in_tokens_used}  out_tokens: {out_tokens_used}"
              f"cost: {self._format_cost()} total[/]"
          )

          return response_text

      except httpx.HTTPError as e:
        status = e.response.status_code
        if status in [429, 500, 502, 503]:
          wait = 2 ** attempt
          console.print(f"[yellow] HTTP {status}. Retrying in {wait} seconds... (attempt {attempt}/{self.max_retries})[/]")
          await asyncio.sleep(wait)
        else:
          #For streaming responses, e.response.text might not be directly available without consuming the stream
          #Read the body if not already done
          try:
            if e.response.is_closed:
              error_details = e.response.text
            else:
              await e.response.aread() #Read the body of the error response
              error_details = e.response.text
          except Exception as body_read_error:
            error_details = f"Failed to read error response body: {body_read_error}"

          console.print(f"[bold red]HTTP {status}:[/] {error_details}")
          return f"[Error {status}]"

      except httpx.TimeoutException:
        console.print(f"[yellow]Timeout on attempt {attempt}/{self.max_retries}[/]")

        if attempt < self.max_retries:
          await asyncio.sleep(2 ** attempt)

      except httpx.RequestError as e:
        console.print(f"[bold red]Network error:[/]")
        return "Could not connect to GROQ. Check your internet connection?"

    return "Max retries exceeded, please try again later..."

  async def get_response(self, user_input:str)->str:
    self._add_to_history("user", user_input)
    messages = [{"role":"system", "content":self.system}] + self._trim_history()
    response_text = await self.stream_response(messages)
    return response_text

  def _print_banner(self) -> None:
        console.print(Panel(
            f"[bold cyan]GROQ Enhanced Chatbot[/]\n"
            f"[dim]Model:[/] [green]{self.model}[/]\n"
            f"[dim]Type [/][bold green]help[/][dim] for commands · [/][bold green]exit[/][dim] to quit[/]",
            title="[bold]Welcome[/]",
            border_style="cyan",
            padding=(1, 4),
        ))

  def _print_help(self) -> None:
        table = Table(
            title="Available Commands",
            box=box.ROUNDED,
            border_style="cyan",
            show_header=True,
            header_style="bold cyan",
        )
        table.add_column("Command", style="bold green", min_width=28)
        table.add_column("Description", style="white")
        rows = [
            ("exit / quit / bye","End the session"),
            ("clear",               "Clear conversation history"),
            ("models",              "List & switch models"),
            ("system",              "Change the system prompt"),
            ("stats",               "Show token & cost stats"),
            ("summary",             "AI summary of the conversation"),
            ("export [json|txt|md]","Export conversation to file"),
            ("help",                "Show this menu"),
        ]
        for cmd, desc in rows:
            table.add_row(cmd, desc)
        console.print(table)

  def _print_stats(self) -> None:
        elapsed = str(datetime.datetime.now() - self.session_start).split(".")[0]
        table = Table(
            title="Session Stats",
            box=box.ROUNDED,
            border_style="magenta",
            show_header=False,
            padding=(0, 2),
        )
        table.add_column("Key",   style="dim", min_width=18)
        table.add_column("Value", style="bold white")
        table.add_row("Model",          self.model)
        table.add_row("Duration",       elapsed)
        table.add_row("Messages",       str(len(self.conversation_history)))
        table.add_row("Tokens in",      f"{self.total_in_tokens:,}")
        table.add_row("Tokens out",     f"{self.total_out_tokens:,}")
        table.add_row("Estimated cost", f"[bold yellow]{self._format_cost()}[/]")
        console.print(table)

  async def _handle_models(self) -> None:
        table = Table(
            title="Available Models",
            box=box.ROUNDED,
            border_style="cyan",
            header_style="bold cyan",
        )
        table.add_column("#",        justify="right", style="dim", width=4)
        table.add_column("Model",    style="white",   min_width=34)
        table.add_column("$/1M in",  justify="right", style="green")
        table.add_column("$/1M out", justify="right", style="yellow")
        table.add_column("Active",   justify="center", width=8)

        for i, m in enumerate(self.AVAILABLE_MODELS, 1):
            p      = MODEL_PRICING.get(m, (0, 0))
            active = "[bold green]✔[/]" if m == self.model else ""
            table.add_row(str(i), m, f"${p[0]:.2f}", f"${p[1]:.2f}", active)
        console.print(table)
        raw = input(
            "[cyan]Enter number to switch model (or press Enter to cancel)[/]"
        ).strip()
        if raw.isdigit():
            idx = int(raw)
            if 1 <= idx <= len(self.AVAILABLE_MODELS):
                self.set_model(self.AVAILABLE_MODELS[idx - 1])
            else:
                console.print("[red]Invalid selection.[/]")

  async def start_conversation(self) -> None:
    self._print_banner()

    while True:
      try:
        user_input = Prompt.ask("[bold magenta]You[/]").strip()

        if not user_input:
          continue

        lower = user_input.lower()

        if lower in ("exit", "quit", "bye"):
          self._print_stats()
          console.print("[bold cyan]Goodbye...[/]")
          break

        if lower == "help":
          self._print_help()
          continue

        if lower == "clear":
          self.conversation_history.clear()
          console.print("[bold green]History cleared[/]")

        if lower == "models":
          await self._handle_models()
          continue

        if lower == "stats":
          self._print_stats()
          continue

        if lower == "system":
          console.print(f"[dim]Current:[/] {self.system}")
          new_prompt = Prompt.ask("[cyan]New system prompt (Enter to keep)[/]", default="").strip()
          if new_prompt:
            self.set_system_prompt(new_prompt)
          continue

        #Normal chat
        await self.get_response(user_input)

      except KeyboardInterrupt:
        console.print("[red]Interrupted, Type `exit`cto quit or keep chatting[/]")
      except EOFError:
        break
      except Exception as e:
        console.print(f"[bold red]Unexpected error:[/] {e}")

In [8]:
async def main()->None:
  chat = GroqChatbot(model="openai/gpt-oss-120b")
  await chat.start_conversation()

In [ ]:
await main()

╭──────────────────────────────────────────────────── Welcome ────────────────────────────────────────────────────╮
│                                                                                                                 │
│    GROQ Enhanced Chatbot                                                                                        │
│    Model: openai/gpt-oss-120b                                                                                   │
│    Type help for commands · exit to quit                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

You: